# Hệ Gợi Ý (Recommender Systems)
## User-Based Collaborative Filtering (User-User CF)
---

### 1. Giới thiệu tổng quan (Introduction)
**Collaborative Filtering (Lọc cộng tác)** là một trong những phương pháp nền tảng và phổ biến nhất trong hệ gợi ý. Ý tưởng cốt lõi của **User-Based Collaborative Filtering (User-User CF)** dựa trên triết lý:
> *"Những người dùng có cùng sở thích/hành vi trong quá khứ thường sẽ tiếp tục có sở thích tương tự nhau trong tương lai."*

### 2. Quy trình thuật toán (Pipeline)
1. **Thu thập và biểu diễn dữ liệu:** Xây dựng ma trận tương tác User-Item (User-Item Matrix) từ tập dữ liệu `goodbooks-10k` (`ratings.csv` và `books.csv`).
2. **Chuẩn hóa điểm đánh giá (Mean-centering / Normalization):** Trừ điểm đánh giá của mỗi người dùng cho điểm trung bình của chính họ nhằm loại bỏ định kiến đánh giá (người chấm khó tính vs người dễ tính).
3. **Tính toán độ tương đồng (User Similarity):** Sử dụng hàm Cosine Similarity trên ma trận đã chuẩn hóa (tương đương với Hệ số tương quan Pearson).
4. **Dự đoán điểm đánh giá (Rating Prediction):** Dự đoán điểm số mà một người dùng $u$ sẽ dành cho cuốn sách $i$ dựa trên trung bình có trọng số từ các người dùng tương đồng nhất (Top-$K$ Neighbors):
$$\hat{r}_{u, i} = \bar{r}_u + \frac{\sum_{v \in N_k(u; i)} \text{sim}(u, v) \cdot (r_{v, i} - \bar{r}_v)}{\sum_{v \in N_k(u; i)} |\text{sim}(u, v)|}$$
5. **Tạo danh sách gợi ý (Top-N Recommendations):** Xếp hạng các cuốn sách chưa đọc theo điểm dự đoán cao nhất.
6. **Đánh giá mô hình (Evaluation):** Đo lường sai số dự đoán bằng các chỉ số **RMSE** (Root Mean Squared Error) và **MAE** (Mean Absolute Error) trên tập Test.

### Bước 1: Cài đặt và Import các thư viện cần thiết

In [ ]:
# Cài đặt các thư viện nếu chưa có trong môi trường
# !pip install numpy pandas scikit-learn matplotlib seaborn scipy

import os
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Cấu hình hiển thị đồ thị
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Đã nạp đầy đủ các thư viện cần thiết!")

### Bước 2: Tải và Khám phá Dữ liệu (EDA - Exploratory Data Analysis)

Tập dữ liệu sử dụng là **Goodbooks-10k**, bao gồm:
- `ratings.csv`: Chứa các bản ghi đánh giá (`user_id`, `book_id`, `rating`).
- `books.csv`: Chứa thông tin chi tiết về sách (`book_id`, `title`, `authors`, `original_publication_year`, `average_rating`, v.v.).

In [ ]:
# Tìm đường dẫn file dữ liệu linh hoạt (hỗ trợ cả chạy từ thư mục gốc hoặc trong subfolder)
possible_rating_paths = ['ratings.csv', 'goodbooks-10k/ratings.csv', '../goodbooks-10k/ratings.csv']
possible_book_paths = ['books.csv', 'goodbooks-10k/books.csv', '../goodbooks-10k/books.csv']

ratings_path = next((p for p in possible_rating_paths if os.path.exists(p)), None)
books_path = next((p for p in possible_book_paths if os.path.exists(p)), None)

if ratings_path is None or books_path is None:
    raise FileNotFoundError("Không tìm thấy file ratings.csv hoặc books.csv. Vui lòng kiểm tra lại đường dẫn!")

print(f"Đang đọc ratings từ: {ratings_path}")
print(f"Đang đọc books từ:   {books_path}")

df_ratings = pd.read_csv(ratings_path)
df_books = pd.read_csv(books_path)

print(f"Tổng số lượt đánh giá ban đầu: {len(df_ratings):,}")
print(f"Tổng số đầu sách: {len(df_books):,}")

In [ ]:
# Hiển thị mẫu dữ liệu ratings
display(df_ratings.head())

# Hiển thị mẫu dữ liệu books (chọn các cột quan trọng)
display(df_books[['book_id', 'title', 'authors', 'original_publication_year', 'average_rating']].head())

#### Phân tích phân phối điểm đánh giá và độ thưa thớt (Sparsity)

In [ ]:
n_users = df_ratings['user_id'].nunique()
n_books = df_ratings['book_id'].nunique()
n_ratings = len(df_ratings)

sparsity = 1.0 - (n_ratings / (n_users * n_books))

print(f"- Số lượng người dùng duy nhất: {n_users:,}")
print(f"- Số lượng sách duy nhất:       {n_books:,}")
print(f"- Tổng số lượt đánh giá:         {n_ratings:,}")
print(f"- Độ thưa ma trận (Sparsity):   {sparsity * 100:.2f}%")

# Trực quan hóa phân phối điểm đánh giá (1-5 sao)
plt.figure(figsize=(8, 4))
rating_counts = df_ratings['rating'].value_counts().sort_index()
bars = plt.bar(rating_counts.index, rating_counts.values, color='#4C72B0', edgecolor='black', alpha=0.85)
plt.title('Phân phối điểm đánh giá (Rating Distribution)', fontsize=13, fontweight='bold')
plt.xlabel('Số sao (Rating)')
plt.ylabel('Số lượt đánh giá')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 20000, f'{yval:,}', ha='center', va='bottom', fontsize=9)
plt.show()

### Bước 3: Lọc dữ liệu mẫu & Chia tập Train / Test

> **Lưu ý về hiệu năng tính toán:**
> Tập dữ liệu Goodbooks-10k có hơn 53.000 người dùng. Việc xây dựng ma trận tương đồng $53.000 \times 53.000$ dạng dense đòi hỏi khoảng $22\text{ GB}$ RAM và thời gian tính toán lớn. 
> Vì vậy, trong đồ án / bài thực hành này, ta lọc ra một tập con gồm các người dùng tích cực nhất (Active Users) và các cuốn sách có nhiều đánh giá nhất để đảm bảo:
> - Có đủ tương tác chồng chéo (overlap) giữa các người dùng để tính toán tương đồng chính xác.
> - Thuật toán chạy nhanh, mượt mà và trực quan trên máy cá nhân.

In [ ]:
# Thiết lập tham số lấy mẫu con phục vụ thực nghiệm
NUM_ACTIVE_USERS = 1500  # Chọn top 1500 người dùng tích cực nhất
NUM_POPULAR_BOOKS = 2000 # Chọn top 2000 sách phổ biến nhất

# Đếm số lượng đánh giá của mỗi user và mỗi book
top_users = df_ratings['user_id'].value_counts().head(NUM_ACTIVE_USERS).index
top_books = df_ratings['book_id'].value_counts().head(NUM_POPULAR_BOOKS).index

# Lọc dữ liệu
df_filtered = df_ratings[
    df_ratings['user_id'].isin(top_users) & 
    df_ratings['book_id'].isin(top_books)
].copy()

print(f"Dữ liệu sau khi lọc:")
print(f"- Số người dùng: {df_filtered['user_id'].nunique()}")
print(f"- Số đầu sách:   {df_filtered['book_id'].nunique()}")
print(f"- Số tương tác:  {len(df_filtered):,}")

In [ ]:
# Chia tập dữ liệu thành 80% Train và 20% Test
train_df, test_df = train_test_split(
    df_filtered, 
    test_size=0.2, 
    random_state=42, 
    stratify=None
)

print(f"Kích thước tập Train: {len(train_df):,} dòng")
print(f"Kích thước tập Test:  {len(test_df):,} dòng")

### Bước 4: Xây dựng Ma trận Tương tác (User-Item Matrix) & Chuẩn hóa điểm đánh giá

#### 1. Định kiến người dùng (User Rating Bias)
- Một số người dùng rất khắt khe: sách hay lắm họ cũng chỉ chấm $3$ hoặc $4$ sao.
- Một số người dùng rất hào phóng: sách bình thường họ cũng chấm $5$ sao.
- Điểm $4$ sao của người khó tính có thể mang ý nghĩa tích cực hơn điểm $4$ sao của người dễ tính.

#### 2. Công thức Chuẩn hóa (Mean-Centering / Mean Normalization)
Tính điểm trung bình $\bar{r}_u$ của người dùng $u$ trên các cuốn sách họ đã đánh giá:
$$\bar{r}_u = \frac{1}{|I_u|} \sum_{i \in I_u} r_{u, i}$$

Ma trận điểm chuẩn hóa $\tilde{R}$:
$$\tilde{r}_{u, i} = \begin{cases} r_{u, i} - \bar{r}_u & \text{nếu user } u \text{ đã đánh giá sách } i \\ 0 & \text{nếu chưa đánh giá} \end{cases}$$

*Điểm dương thể hiện user thích hơn mức bình thường, điểm âm thể hiện user không thích bằng mức bình thường, điểm 0 thể hiện chưa biết hoặc đúng bằng mức trung bình.*

In [ ]:
# Tạo ánh xạ ID (mapping) liên tục từ 0 -> N-1 cho User và Book
unique_users = sorted(train_df['user_id'].unique())
unique_books = sorted(train_df['book_id'].unique())

user2idx = {u: i for i, u in enumerate(unique_users)}
idx2user = {i: u for i, u in enumerate(unique_users)}

book2idx = {b: i for i, b in enumerate(unique_books)}
idx2book = {i: b for i, b in enumerate(unique_books)}

n_u = len(unique_users)
n_b = len(unique_books)

print(f"Ma trận tương tác có kích thước: {n_u} users x {n_b} books")

In [ ]:
# Tạo User-Item Rating Matrix gốc (R)
R = np.zeros((n_u, n_b), dtype=np.float32)
for row in train_df.itertuples():
    if row.user_id in user2idx and row.book_id in book2idx:
        u_idx = user2idx[row.user_id]
        b_idx = book2idx[row.book_id]
        R[u_idx, b_idx] = row.rating

# Tính điểm trung bình của mỗi user (chỉ tính trên các item đã được đánh giá)
user_ratings_count = np.count_nonzero(R, axis=1)
user_ratings_sum = np.sum(R, axis=1)

# Tránh chia cho 0 nếu user chưa đánh giá cuốn nào
user_means = np.divide(
    user_ratings_sum, 
    user_ratings_count, 
    out=np.zeros_like(user_ratings_sum, dtype=float), 
    where=user_ratings_count != 0
)

# Tạo ma trận chuẩn hóa R_normalized bằng cách trừ đi user_means
R_norm = np.zeros_like(R)
for u in range(n_u):
    rated_indices = np.where(R[u, :] > 0)[0]
    R_norm[u, rated_indices] = R[u, rated_indices] - user_means[u]

print(f"Ví dụ 5 người dùng đầu tiên:")
for u in range(5):
    print(f"- User ID: {idx2user[u]} | Số sách đã đánh giá: {user_ratings_count[u]:3d} | Điểm TB (Mean): {user_means[u]:.2f}")

### Bước 5: Tính toán Ma trận Tương đồng giữa các Người dùng (User Similarity Matrix)

Độ tương đồng Cosine giữa hai người dùng $u$ và $v$ trên các vector điểm đã chuẩn hóa:
$$\text{sim}(u, v) = \cos(\mathbf{\tilde{r}}_u, \mathbf{\tilde{r}}_v) = \frac{\mathbf{\tilde{r}}_u \cdot \mathbf{\tilde{r}}_v}{\|\mathbf{\tilde{r}}_u\|_2 \|\mathbf{\tilde{r}}_v\|_2}$$

>*Lưu ý quan trọng:* Khi áp dụng Cosine Similarity trên các vector đã mean-centered, công thức này tương đương với **Hệ số tương quan Pearson (Pearson Correlation Coefficient)**, đo lường sự tương quan trong xu hướng đánh giá của hai người dùng.

In [ ]:
# Tính toán độ tương đồng Cosine trên ma trận chuẩn hóa
start_time = time.time()
user_similarity = cosine_similarity(R_norm)
print(f"Thời gian tính toán ma trận tương đồng: {time.time() - start_time:.3f} giây")

# Độ tương đồng của một người với chính mình bằng 1.0, ta có thể đặt đường chéo chính bằng 0 khi tìm láng giềng
np.fill_diagonal(user_similarity, 0.0)

print(f"Kích thước ma trận tương đồng người dùng: {user_similarity.shape}")

In [ ]:
# Trực quan hóa ma trận tương đồng cho 30 người dùng đầu tiên
plt.figure(figsize=(9, 7))
sample_sim = user_similarity[:30, :30]
sns.heatmap(sample_sim, cmap='coolwarm', center=0, cbar_kws={'label': 'Độ tương đồng (Cosine Sim)'})
plt.title('Heatmap độ tương đồng giữa 30 người dùng mẫu', fontsize=12, fontweight='bold')
plt.xlabel('Chỉ số người dùng')
plt.ylabel('Chỉ số người dùng')
plt.show()

### Bước 6: Xây dựng Lớp Mô hình Hoàn chỉnh (UserBasedCollaborativeFiltering)

Chúng ta đóng gói toàn bộ logic vào một lớp đối tượng `UserBasedCollaborativeFiltering` chuẩn phong cách Scikit-Learn với các phương thức:
- `fit(train_ratings)`: Huấn luyện mô hình (lập ma trận, chuẩn hóa, tính similarity).
- `predict_rating(user_id, book_id, k)`: Dự đoán số sao cho 1 cặp user-item dựa trên Top-$k$ láng giềng.
- `recommend(user_id, top_n, k)`: Đưa ra Top-$N$ cuốn sách gợi ý tốt nhất mà user chưa đọc.

In [ ]:
class UserBasedCollaborativeFiltering:
    def __init__(self, k_neighbors=20, min_similarity=0.0):
        """
        k_neighbors: Số lượng láng giềng gần nhất (Top-K tương đồng) được dùng để suy luận.
        min_similarity: Ngưỡng tương đồng tối thiểu (thường chọn >= 0 để loại trừ người có gu đối nghịch).
        """
        self.k_neighbors = k_neighbors
        self.min_similarity = min_similarity
        self.user2idx = {}
        self.idx2user = {}
        self.book2idx = {}
        self.idx2book = {}
        self.user_means = None
        self.global_mean = 3.0
        self.R = None
        self.R_norm = None
        self.user_similarity = None

    def fit(self, df_train):
        """Huấn luyện mô hình từ DataFrame gồm các cột: user_id, book_id, rating"""
        unique_users = sorted(df_train['user_id'].unique())
        unique_books = sorted(df_train['book_id'].unique())

        self.user2idx = {u: i for i, u in enumerate(unique_users)}
        self.idx2user = {i: u for i, u in enumerate(unique_users)}
        self.book2idx = {b: i for i, b in enumerate(unique_books)}
        self.idx2book = {i: b for i, b in enumerate(unique_books)}

        n_u = len(unique_users)
        n_b = len(unique_books)
        self.global_mean = df_train['rating'].mean()

        # Tạo ma trận R
        self.R = np.zeros((n_u, n_b), dtype=np.float32)
        for row in df_train.itertuples():
            if row.user_id in self.user2idx and row.book_id in self.book2idx:
                u = self.user2idx[row.user_id]
                b = self.book2idx[row.book_id]
                self.R[u, b] = row.rating

        # Tính điểm trung bình của mỗi user
        counts = np.count_nonzero(self.R, axis=1)
        sums = np.sum(self.R, axis=1)
        self.user_means = np.divide(
            sums, counts, 
            out=np.full(n_u, self.global_mean, dtype=float), 
            where=counts > 0
        )

        # Chuẩn hóa ma trận (Mean-centered)
        self.R_norm = np.zeros_like(self.R)
        for u in range(n_u):
            rated_items = np.where(self.R[u, :] > 0)[0]
            self.R_norm[u, rated_items] = self.R[u, rated_items] - self.user_means[u]

        # Tính ma trận tương đồng người dùng
        self.user_similarity = cosine_similarity(self.R_norm)
        np.fill_diagonal(self.user_similarity, 0.0)
        print(f"Huấn luyện xong UserCF: {n_u} users, {n_b} books, global_mean = {self.global_mean:.2f}")
        return self

    def predict_rating(self, user_id, book_id):
        """
        Dự đoán điểm mà user_id sẽ dành cho book_id.
        Áp dụng công thức có trọng số tương đồng từ Top-K láng giềng đã đánh giá cuốn sách này.
        """
        # Nếu user chưa từng xuất hiện trong tập train (Cold-start user)
        if user_id not in self.user2idx:
            return self.global_mean
        
        u = self.user2idx[user_id]
        u_mean = self.user_means[u]

        # Nếu cuốn sách chưa từng xuất hiện trong tập train (Cold-start book)
        if book_id not in self.book2idx:
            return u_mean

        b = self.book2idx[book_id]

        # Tìm tất cả người dùng khác đã đánh giá cuốn sách này
        other_users = np.where(self.R[:, b] > 0)[0]
        # Loại bỏ chính người dùng u nếu có
        other_users = other_users[other_users != u]

        if len(other_users) == 0:
            return u_mean

        # Lấy độ tương đồng giữa u và những người dùng đã đánh giá sách b
        sims = self.user_similarity[u, other_users]

        # Chỉ giữ lại những người có độ tương đồng > min_similarity
        valid_mask = sims > self.min_similarity
        if not np.any(valid_mask):
            return u_mean

        filtered_users = other_users[valid_mask]
        filtered_sims = sims[valid_mask]

        # Chọn ra Top-K láng giềng có độ tương đồng cao nhất
        if len(filtered_sims) > self.k_neighbors:
            top_k_indices = np.argsort(filtered_sims)[-self.k_neighbors:]
            filtered_users = filtered_users[top_k_indices]
            filtered_sims = filtered_sims[top_k_indices]

        # Tính độ lệch đã chuẩn hóa của các láng giềng cho cuốn sách b
        # R_norm[v, b] chính là (r_{v, b} - bar{r}_v)
        norm_ratings = self.R_norm[filtered_users, b]

        # Tổng trọng số tương đồng
        sim_sum = np.sum(np.abs(filtered_sims))
        if sim_sum == 0:
            return u_mean

        # Công thức dự đoán điểm:
        # hat{r}_{u, b} = bar{r}_u + sum(sim(u, v) * (r_{v, b} - bar{r}_v)) / sum(|sim(u, v)|)
        pred = u_mean + (np.sum(filtered_sims * norm_ratings) / sim_sum)

        # Giới hạn trong khoảng thang điểm [1.0, 5.0]
        return np.clip(pred, 1.0, 5.0)

    def recommend(self, user_id, top_n=10):
        """
        Gợi ý Top-N cuốn sách phù hợp nhất cho user_id trong số các sách họ chưa từng đọc.
        """
        if user_id not in self.user2idx:
            print(f"User {user_id} không có trong dữ liệu huấn luyện!")
            return pd.DataFrame()

        u = self.user2idx[user_id]
        # Danh sách sách user đã đánh giá
        rated_book_indices = set(np.where(self.R[u, :] > 0)[0])

        # Dự đoán điểm cho tất cả các cuốn sách user chưa đọc
        unrated_book_indices = [b for b in range(len(self.idx2book)) if b not in rated_book_indices]

        predictions = []
        for b in unrated_book_indices:
            book_id = self.idx2book[b]
            predicted_score = self.predict_rating(user_id, book_id)
            predictions.append((book_id, predicted_score))

        # Sắp xếp giảm dần theo điểm dự đoán
        predictions.sort(key=lambda x: x[1], reverse=True)
        top_predictions = predictions[:top_n]

        res_df = pd.DataFrame(top_predictions, columns=['book_id', 'predicted_rating'])
        return res_df

### Bước 7: Đánh giá Mô hình trên Tập Test (Evaluation: RMSE & MAE)

Chúng ta đo lường độ chính xác dự đoán của thuật toán dựa trên:
1. **Root Mean Squared Error (RMSE):**
$$\text{RMSE} = \sqrt{\frac{1}{|T|} \sum_{(u, i) \in T} (r_{u, i} - \hat{r}_{u, i})^2}$$

2. **Mean Absolute Error (MAE):**
$$\text{MAE} = \frac{1}{|T|} \sum_{(u, i) \in T} |r_{u, i} - \hat{r}_{u, i}|$$

In [ ]:
# Khởi tạo và fit mô hình
model = UserBasedCollaborativeFiltering(k_neighbors=25, min_similarity=0.0)
model.fit(train_df)

# Đánh giá trên một tập mẫu test để đo tốc độ và sai số
test_sample = test_df.sample(n=min(3000, len(test_df)), random_state=42)

actual_ratings = []
predicted_ratings = []

start_eval = time.time()
for row in test_sample.itertuples():
    pred = model.predict_rating(row.user_id, row.book_id)
    actual_ratings.append(row.rating)
    predicted_ratings.append(pred)

eval_time = time.time() - start_eval
rmse = math.sqrt(mean_squared_error(actual_ratings, predicted_ratings))
mae = mean_absolute_error(actual_ratings, predicted_ratings)

print(f"=== KẾT QUẢ ĐÁNH GIÁ (TEST EVALUATION) ===")
print(f"- Số lượng mẫu đánh giá: {len(test_sample):,} samples")
print(f"- Thời gian tính toán:    {eval_time:.2f} giây")
print(f"- RMSE:                  {rmse:.4f}")
print(f"- MAE:                   {mae:.4f}")

#### Thực nghiệm: Ảnh hưởng của số lượng láng giềng $K$ đến độ chính xác (Hyperparameter Tuning for $K$)

In [ ]:
k_values = [5, 10, 20, 30, 50]
rmse_scores = []
mae_scores = []

small_test = test_sample.head(1000)

for k in k_values:
    model_k = UserBasedCollaborativeFiltering(k_neighbors=k, min_similarity=0.0)
    model_k.fit(train_df)
    
    preds = [model_k.predict_rating(row.user_id, row.book_id) for row in small_test.itertuples()]
    actuals = small_test['rating'].values
    
    k_rmse = math.sqrt(mean_squared_error(actuals, preds))
    k_mae = mean_absolute_error(actuals, preds)
    
    rmse_scores.append(k_rmse)
    mae_scores.append(k_mae)
    print(f"K = {k:2d} | RMSE = {k_rmse:.4f} | MAE = {k_mae:.4f}")

# Vẽ biểu đồ so sánh RMSE theo giá trị K
plt.figure(figsize=(8, 4))
plt.plot(k_values, rmse_scores, marker='o', linewidth=2, color='#2b5c8f', label='RMSE')
plt.plot(k_values, mae_scores, marker='s', linewidth=2, color='#d95f02', label='MAE')
plt.title('Độ lỗi (Error) theo số lượng láng giềng K (Top-K Neighbors)', fontsize=12, fontweight='bold')
plt.xlabel('Số láng giềng K')
plt.ylabel('Độ lỗi')
plt.xticks(k_values)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Bước 8: Trực quan hóa & Đưa ra gợi ý sách thực tế (Demonstration)

Dưới đây, ta chọn một người dùng cụ thể trong hệ thống để xem:
1. **Lịch sử đọc của người dùng:** Những cuốn sách mà họ từng chấm điểm cao nhất (gu đọc sách của họ).
2. **Danh sách Top-10 cuốn sách gợi ý:** Kiểm tra xem sách được gợi ý có phản ánh đúng gu sở thích của họ không.

In [ ]:
# Chọn một user_id ngẫu nhiên có nhiều đánh giá trong tập train
sample_user_id = train_df['user_id'].value_counts().index[10]
print(f"Đang thực hiện gợi ý cho User ID: {sample_user_id}")

# 1. Hiển thị các cuốn sách user đã đánh giá cao nhất (5 sao)
user_history = df_ratings[df_ratings['user_id'] == sample_user_id].merge(
    df_books[['book_id', 'title', 'authors', 'original_publication_year', 'average_rating']],
    on='book_id'
).sort_values(by='rating', ascending=False)

print(f"\n--- TOP SÁCH USER {sample_user_id} ĐÃ ĐỌC VÀ ĐÁNH GIÁ CAO NHẤT ---")
display(user_history[['title', 'authors', 'original_publication_year', 'rating', 'average_rating']].head(7))

# 2. Tạo gợi ý Top-10 bằng mô hình User-Based CF
recommendations = model.recommend(user_id=sample_user_id, top_n=10)

# Kết hợp thông tin sách từ df_books
rec_books = recommendations.merge(
    df_books[['book_id', 'title', 'authors', 'original_publication_year', 'average_rating']],
    on='book_id'
)

print(f"\n--- TOP 10 SÁCH ĐƯỢC GỢI Ý CHO USER {sample_user_id} ---")
display(rec_books[['title', 'authors', 'original_publication_year', 'predicted_rating', 'average_rating']])

### Bước 9: Tổng kết và Đánh giá Ưu / Nhược điểm của User-Based CF

| Ưu điểm (Pros) | Nhược điểm (Cons) | Giải pháp cải tiến (Advanced Solutions) |
| :--- | :--- | :--- |
| **Dễ hiểu & Trực quan:** Rất dễ giải thích cho người dùng: *"Gợi ý vì những người giống bạn cũng thích sách này"*. | **Độ phức tạp tính toán cao:** Số người dùng $U$ thường rất lớn. Tính ma trận $U \times U$ tốn bộ nhớ $O(U^2)$. | **Item-Based CF:** Trong thực tế, số lượng sách/sản phẩm thường ít biến động hơn số lượng người dùng. |
| **Không cần trích xuất đặc trưng nội dung:** Hoạt động tốt mà không cần phân tích văn bản sách hay ảnh bìa. | **Vấn đề khởi đầu lạnh (Cold-Start):** Người dùng mới hoặc sách mới chưa có tương tác không thể tính toán được. | **Hybrid System:** Kết hợp Content-Based Filtering cho Cold-Start và CF khi đã có tương tác. |
| **Khám phá bất ngờ (Serendipity):** Có thể gợi ý các chủ đề mới lạ mà người dùng chưa từng nghĩ tới nhưng cộng đồng cùng gu lại thích. | **Ma trận thưa thớt (Data Sparsity):** Khi người dùng chỉ đánh giá 0.1% số sách, khả năng tìm thấy người có cùng tập sách trùng nhau rất thấp. | **Matrix Factorization (SVD, ALS, NMF):** Nén không gian thưa thớt sang các chiều ẩn (Latent Factors) như SVD. |